# M18 — NFL Scouting Value (Colab)

**Status: tasks T15–T17 authored — never executed by the author.**

## A. Introduction

- **Question.** Do validated Combine tracking features add independently predictive information about NFL movement beyond official Combine testing (M1)?
- **Design.** One cohort, one fold system (5 athlete-disjoint folds), one target per comparison, the official ridge engine. Ladder B0 position · B1 M1 · B2 +simple · B3 +M14 validated · B4 +STFT · B5 +LSTM/GRU · B6 +fusion · B7 +G17 signatures. A rung runs only when its upstream gate allows it; otherwise it is **blocked** with the reason.
- **Pre-registration.** The primary target and the primary family (eligible B2–B7 vs B1, Holm) are written to `manifests/preregistration.json` **before** any outcome-bearing step and never overwritten.
- **Studies.** A ladder · B position-specific models · C drill → task specificity · D stopwatch-twin scouting cases (selected without outcomes) · E controls, random label, permutation null, sensitivity · F evidence registry, **G18**, final report, executive summary.
- **A rigorous negative result is an acceptable outcome**; every verdict lists replication on new draft classes as required. No injury, force, reaction-time or causal-development claims.
- **Inputs (read only).** M14, M15, M16, M17 runs; bundle via the experiment.

## B. Setup

In [ ]:
# Setup 1 — mount Drive, install nflbowl (Colab PyTorch / CUDA kept)
from google.colab import drive
drive.mount('/content/drive')
REPO = 'git+https://github.com/mybenkhadda/NFL_bowl.git@main'
%pip install -q {REPO}
%pip install -q --upgrade --force-reinstall --no-deps {REPO}
%pip install -q markdown

In [ ]:
# Setup 2 — imports (after a reinstall: Runtime → Restart session, then continue from here)
import nflbowl
print('nflbowl', nflbowl.__version__)
assert tuple(int(x) for x in nflbowl.__version__.split('.')[:2]) >= (0, 7)
import json
import polars as pl
from IPython.display import Markdown, display
from nflbowl.lab import handoff
from nflbowl.lab import status as S
from nflbowl.lab.m18 import config as m18cfg, pipeline as m18pipe, plots as m18plots

In [ ]:
# Setup 3 — configuration (the ONLY cell to edit). Changing primary_target after the first run is reported as a pre-registration conflict.
CFG = {
    'experiment_name': 'm18_scouting_transfer_v1',
    'bundle_dir': '/content/drive/MyDrive/nfl-big-data-bowl-2027/bundles/m12f_v1_clean',
    'drive_root': '/content/drive/MyDrive/NFLBowl',
    'primary_target': 'T0',            # nfl.q90_shared unless a recorded G16-A decision says otherwise
    'secondary_targets': 'eligible',   # M16-frozen eligible task targets
    'allow_descriptive': False,
    'n_perm': 200,
    'pseudonym_salt': 'change-me',
    'seed': 42,
}
cfg = m18cfg.resolve(CFG)
run = m18cfg.M18Run(cfg.run_dir, cfg)
run.snapshot_config(cfg, handoff.environment(cfg.m13()))
print('run dir', run.root, '· fingerprint', cfg.fingerprint())
display(run.ledger.table())

In [ ]:
# Setup 4 — experiment (official engine, folds) + analysis frame, rung eligibility and the immutable pre-registration
def step_open(ctx):
    return handoff.open_experiment(cfg.m13(), run)

EXP = S.run_step(run, run.ledger, 'E18.open', step_open, note='open the M18 experiment on the official bundle / science')
if EXP is None:
    raise SystemExit('the experiment could not be opened — see the status ledger and logs/')

def step_prep(ctx):
    return m18pipe.prepare(cfg, EXP, run, ctx)

P = S.run_step(run, run.ledger, 'E18.prepare', step_prep, note='analysis frame + rung eligibility + pre-registration', requires=['E18.open'])
if P is None:
    raise SystemExit('preparation failed — see the reason (M16 outputs are required)')
display(P['rung_frame'])
print('pre-registration:', {k: P['prereg'].get(k) for k in ('primary_target', 'primary_family', 'conflict', 'written_at')})
print('targets:', P['targets'])

## E. Study A — augmentation ladder (E18.A1)

In [ ]:
def step_a(ctx):
    return m18pipe.run_a(cfg, EXP, P, run, ctx)

A = S.run_step(run, run.ledger, 'E18.A', step_a, note='ladder B0–B7 on every target', requires=['E18.prepare'])
if A is not None:
    pr = A['primary']
    display(pr['metrics'].select('model', 'n_athletes', 'rmse', 'r2', 'spearman'))
    display(pr['paired'].select('model', 'reference', 'delta_rmse', 'delta_lo', 'delta_hi', 'delta_pct', 'p_holm_primary', 'precise_null', 'verdict'))
    display(pr['status'])
    print('best rung (largest ΔRMSE vs M1, used for controls and twins):', A['best'])

In [ ]:
if A is not None and A['primary']['paired'].height:
    fig, dat = m18plots.ladder_forest(A['primary']['paired'], 'M1', cfg.margin_pct)
    run.figure(fig, 'M18_A_01_ladder_forest', data=dat, question='Does tracking add information beyond official Combine testing?',
               caption='ΔRMSE vs M1 (> 0 = better), 95 % fold-stratified athlete bootstrap; grey band = practical margin.')
    fig, dat = m18plots.fold_lines(A['primary']['folds'])
    run.figure(fig, 'M18_A_02_fold_lines', data=dat, question='Is any gain stable across folds?', caption='ΔRMSE vs M1 per outer fold.')

## Study B — position-specific models · Study C — drill → task specificity

In [ ]:
def step_b(ctx):
    return m18pipe.run_b(cfg, EXP, P, A, run, ctx)

B = S.run_step(run, run.ledger, 'E18.B', step_b, note='pooled vs separate within-position models', requires=['E18.A'])
if B is not None:
    display(B['verdict']); display(B['table'])

def step_c(ctx):
    return m18pipe.run_c(cfg, P, run, ctx)

C = S.run_step(run, run.ledger, 'E18.C', step_c, note='drill × task specificity', requires=['E18.prepare'])
if C is not None:
    display(C['contrast'])
    fig, dat = m18plots.specificity_heatmap(C['long'])
    run.figure(fig, 'M18_C_01_specificity', data=dat, question='Does each drill transfer to its matching NFL task?', caption='Partial Spearman | M1 + position; outlined = pre-declared match.')

## Study D — stopwatch twins and scouting cases (selected without outcomes)

In [ ]:
def step_d(ctx):
    return m18pipe.run_d(cfg, P, A, run, ctx)

D = S.run_step(run, run.ledger, 'E18.D', step_d, note='twins, pooled contrast, case cards', requires=['E18.A'])
if D is not None:
    display(D['summary'])
    print('composite built from:', D['composite_columns'])
    print('case cards:', D['files'])
    fig, dat = m18plots.twin_scatter(D['long'])
    run.figure(fig, 'M18_D_01_twins', data=dat, question='Does tracking separate same-stopwatch prospects in the NFL?', caption='Per twin pair: Δcomposite vs Δoutcome.')
    if D['files']:
        display(Markdown(open(D['files'][0], encoding='utf-8').read()))

## Study E — robustness, negative controls, permutation null

In [ ]:
def step_e(ctx):
    return m18pipe.run_e(cfg, EXP, P, A, run, ctx)

E = S.run_step(run, run.ledger, 'E18.E', step_e, note='controls + random label + permutation + sensitivity', requires=['E18.A'])
if E is not None:
    display(E['controls']); display(E['random_label']); print('permutation:', E['permutation']); display(E['sensitivity'])
    fig, dat = m18plots.controls_panel(E['controls'], E['random_label'], E['permutation'])
    run.figure(fig, 'M18_E_01_controls', data=dat, question='Do controls stay at zero?', caption='Negative controls and random-label ΔRMSE vs M1.')

## F. Evidence registry, G18, report and executive summary

In [ ]:
def step_f(ctx):
    if A is None:
        raise S.ResearchBlocked(S.BLOCKED_MISSING_INPUT, 'ladder not available this session', missing=['E18.A'])
    return m18pipe.run_f(cfg, P, A, B, C, D, E, run, ctx)

F = S.run_step(run, run.ledger, 'G18', step_f, note='evidence registry + G18 + reports', requires=['E18.A'])
if F is not None:
    display(F['evidence']); display(F['gate'])
    display(Markdown(F['executive']))
    fig, dat = m18plots.executive_figure(F['evidence'], A['primary']['paired'], P['targets'][cfg.primary_target])
    run.figure(fig, 'M18_F_01_executive_figure', data=dat, question='Should tracking change how prospects are ranked?', caption='ΔRMSE vs M1 per rung coloured by G18 verdict.')
    fig, dat = m18plots.gate_matrix(F['gate'].with_columns(pl.lit('M18').alias('family')), 'G18')
    run.figure(fig, 'M18_F_02_gate', data=dat, question='Which families earn scouting relevance?', caption='G18 verdicts.')

## G. Persistence and hand-off

In [ ]:
run.write_status()
run.save_markdown(run.handoff_markdown(), 'm18_handoff')
display(run.ledger.table())
for rel in ('manifests/preregistration.json', 'predictions/oof_long.parquet', 'tables/ladder', 'tables/position', 'tables/specificity', 'tables/scouting',
            'tables/controls', 'tables/robustness', 'tables/synthesis', 'reports/markdown/m18_final_report.md', 'reports/markdown/executive_summary.md',
            'reports/athlete_profiles', 'figures/png'):
    p = run.root / rel
    print(('✓ ' if p.exists() else '· ') + str(p))
print('SEND BACK: manifests/stages.json, manifests/m18_status.json, manifests/preregistration.json, logs/*_traceback.txt, tables/ladder/paired.csv,')
print('           tables/ladder/status.csv, tables/controls/negative_controls.csv, tables/synthesis/evidence.csv, tables/synthesis/m18_gate.csv,')
print('           reports/markdown/m18_final_report.md, reports/markdown/executive_summary.md')